# 09 HPAI spatiotemporal feature evaluation v5 leakage-safe + auto Drive save

このノートブックは，`06_rolling_riskmap_evaluation` と同じディレクトリ構造・同じ評価設計を維持しつつ，
追加の時空間ラグ特徴量を作成してrolling risk-map評価を行う．

v3で発生した不自然な完全的中を避けるため，以下を明示する．

- `num_birds_sum` は説明変数から除外する．
- `outbreak_count`, `outbreak_binary`, `target`, `y_lead_*` は説明変数から除外する．
- 評価splitは06と同じ年度区切り，すなわち4月始まりにする．
- 目的変数は06と同じ `outbreak_binary` を使う．
- 追加特徴量は過去週だけから作る．同週・未来週の発生情報は使わない．

In [1]:
# ============================================================
# 0. Google Drive mount and basic settings
# ============================================================
import os
import glob
import gc
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.neighbors import BallTree
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import joblib

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from google.colab import drive
    if os.path.exists('/content/drive/MyDrive'):
        print('Google Drive is already visible at /content/drive/MyDrive')
    else:
        drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped or failed:', repr(e))

NOTEBOOK_VERSION = '09_add_spatiotemporal_environment_features_same_paths_v5_auto_save_to_drive'
print('NOTEBOOK VERSION:', NOTEBOOK_VERSION)

# 06/07/08と同じパス
PROC_DIR = '/content/drive/MyDrive/avian_influenza_project/processed'
MODEL_DIR = f'{PROC_DIR}/model_outputs_riskmap_eval'
RESULT_DIR = MODEL_DIR
# 最終出力を明示的に集約するGoogle Drive上のフォルダ
FINAL_OUTPUT_DIR = f'{MODEL_DIR}/09_final_outputs_v5_auto_save_to_drive'

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)

# メモリ制御
RANDOM_STATE = 42
N_ESTIMATORS = 300
MAX_DEPTH = 12
MIN_SAMPLES_LEAF = 10
N_JOBS = -1

# 近傍特徴量設定．重い場合は [30] にする．
RADII_KM = [30, 50]
WINDOWS_WEEKS = [1, 2, 4, 8, 12]

print('PROC_DIR:', PROC_DIR)
print('MODEL_DIR:', MODEL_DIR)
print('RESULT_DIR:', RESULT_DIR)
print('RADII_KM:', RADII_KM)
print('WINDOWS_WEEKS:', WINDOWS_WEEKS)

Mounted at /content/drive
NOTEBOOK VERSION: 09_add_spatiotemporal_environment_features_same_paths_v5_auto_save_to_drive
PROC_DIR: /content/drive/MyDrive/avian_influenza_project/processed
MODEL_DIR: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval
RESULT_DIR: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval
RADII_KM: [30, 50]
WINDOWS_WEEKS: [1, 2, 4, 8, 12]


In [2]:
# ============================================================
# 1. Load model-ready data: same priority as 06
# ============================================================
MODEL_INPUT_PRIMARY = f'{PROC_DIR}/hpai_weekly_grid_panel_for_model.parquet'
MODEL_INPUT_FALLBACK = f'{PROC_DIR}/hpai_weekly_grid_panel_with_weather_model_ready.parquet'
MODEL_INPUT_FALLBACK2 = f'{PROC_DIR}/hpai_weekly_grid_panel_with_weather.parquet'

candidate_paths = [
    MODEL_INPUT_PRIMARY,
    MODEL_INPUT_FALLBACK,
    MODEL_INPUT_FALLBACK2,
]
existing = [p for p in candidate_paths if os.path.exists(p)]

if existing:
    data_path = existing[0]
else:
    print('指定ファイルが見つからないため，PROC_DIR配下を探索します．')
    patterns = [
        f'{PROC_DIR}/**/*for_model*.parquet',
        f'{PROC_DIR}/**/*model_ready*.parquet',
        f'{PROC_DIR}/**/*with_weather*.parquet',
        f'{PROC_DIR}/**/*.parquet',
        f'{PROC_DIR}/**/*for_model*.csv',
        f'{PROC_DIR}/**/*model_ready*.csv',
        f'{PROC_DIR}/**/*with_weather*.csv',
        f'{PROC_DIR}/**/*.csv',
    ]
    found = []
    for pat in patterns:
        found.extend(glob.glob(pat, recursive=True))
    found = sorted(set(found))
    print('候補ファイル:')
    for f in found[:50]:
        print(f)
    if not found:
        raise FileNotFoundError('モデル用データが見つかりません．04前処理ノートブックで保存してください．')
    data_path = found[0]

print('読み込みファイル:', data_path)

if data_path.lower().endswith('.csv'):
    df = pd.read_csv(data_path)
else:
    df = pd.read_parquet(data_path)

df['week_start'] = pd.to_datetime(df['week_start'])
if 'grid_id' in df.columns:
    df['grid_id'] = df['grid_id'].astype(str)

print('df shape:', df.shape)
print('period:', df['week_start'].min(), 'to', df['week_start'].max())
print('grid:', df['grid_id'].nunique() if 'grid_id' in df.columns else 'grid_id column not found')
print('columns:', len(df.columns))
display(df.head())

読み込みファイル: /content/drive/MyDrive/avian_influenza_project/processed/hpai_weekly_grid_panel_for_model.parquet
df shape: (1641809, 34)
period: 2020-08-31 00:00:00 to 2026-05-18 00:00:00
grid: 5491
columns: 34


,grid_id,week_start,year,week,year_week,month,outbreak_count,outbreak_binary,num_birds_sum,weekofyear,...,neighbor_outbreak_binary,neighbor_outbreak_count_past_1w,neighbor_outbreak_count_past_2w,neighbor_outbreak_count_past_4w,neighbor_outbreak_count_past_8w,grid_lat,grid_lon,temp_mean_c,temp_min_c,temp_max_c
0,G000015,2020-08-31,2020,36,2020-36,8,0,0,0.0,36,...,0,0,0,0,0,24.250542,123.786006,27.533695,25.785553,29.286774
1,G000015,2020-09-07,2020,37,2020-37,9,0,0,0.0,37,...,0,0,0,0,0,24.250542,123.786006,27.620443,25.915192,29.147369
2,G000015,2020-09-14,2020,38,2020-38,9,0,0,0.0,38,...,0,0,0,0,0,24.250542,123.786006,28.277975,26.086334,29.509674
3,G000015,2020-09-21,2020,39,2020-39,9,0,0,0.0,39,...,0,0,0,0,0,24.250542,123.786006,26.583055,23.120026,27.923492
4,G000015,2020-09-28,2020,40,2020-40,9,0,0,0.0,40,...,0,0,0,0,0,24.250542,123.786006,26.359789,24.930328,27.698395


In [3]:
# ============================================================
# 2. Target check: same target as 06
# ============================================================
target_col = 'outbreak_binary'

if target_col not in df.columns:
    if 'outbreak_count' not in df.columns:
        raise ValueError('outbreak_binary も outbreak_count もありません．目的変数を作れません．')
    print('outbreak_binary がないため，outbreak_count > 0 から作成します．')
    df[target_col] = (df['outbreak_count'].fillna(0) > 0).astype(int)

if 'grid_id' not in df.columns:
    raise ValueError('grid_id がありません．')
if 'week_start' not in df.columns:
    raise ValueError('week_start がありません．')

df[target_col] = df[target_col].fillna(0).astype(int)

print(df[target_col].value_counts(dropna=False))
print('overall event rate:', df[target_col].mean())
print('event rows:', int(df[target_col].sum()))

if 'outbreak_count' in df.columns:
    print('total outbreak_count:', df['outbreak_count'].sum())

outbreak_binary
0    1641618
1        191
Name: count, dtype: int64
overall event rate: 0.00011633509135350093
event rows: 191
total outbreak_count: 223


In [4]:
# ============================================================
# 3. Utility: find lon/lat columns and make season columns
# ============================================================
def find_lon_lat_cols(frame):
    lon_candidates = ['centroid_lon', 'grid_lon', 'lon', 'longitude']
    lat_candidates = ['centroid_lat', 'grid_lat', 'lat', 'latitude']
    lon_col = next((c for c in lon_candidates if c in frame.columns), None)
    lat_col = next((c for c in lat_candidates if c in frame.columns), None)
    return lon_col, lat_col

lon_col, lat_col = find_lon_lat_cols(df)
print('lon_col:', lon_col, 'lat_col:', lat_col)

if lon_col is None or lat_col is None:
    raise ValueError('緯度経度列が見つかりません．近傍特徴量を作れません．')

if 'weekofyear' not in df.columns:
    df['weekofyear'] = df['week_start'].dt.isocalendar().week.astype(int)
if 'month' not in df.columns:
    df['month'] = df['week_start'].dt.month.astype(int)
if 'sin_week' not in df.columns:
    df['sin_week'] = np.sin(2 * np.pi * df['weekofyear'] / 52.0)
if 'cos_week' not in df.columns:
    df['cos_week'] = np.cos(2 * np.pi * df['weekofyear'] / 52.0)

# 数値型を軽量化
for c in df.columns:
    if c not in ['grid_id', 'week_start'] and pd.api.types.is_float_dtype(df[c]):
        df[c] = df[c].astype('float32')

lon_col: grid_lon lat_col: grid_lat


In [5]:
# ============================================================
# 4. Create strict past-only spatiotemporal outbreak features
# ============================================================
# 既に06系の特徴量がある場合も，ここでv4専用の列名として作り直す．

df = df.sort_values(['grid_id', 'week_start']).reset_index(drop=True)

# Grid and week index
grid = (df[['grid_id', lat_col, lon_col]]
        .drop_duplicates('grid_id')
        .sort_values('grid_id')
        .reset_index(drop=True))
grid_ids = grid['grid_id'].astype(str).tolist()
grid_index = {g:i for i,g in enumerate(grid_ids)}

weeks = sorted(df['week_start'].unique())
week_index = {w:i for i,w in enumerate(weeks)}

n_weeks = len(weeks)
n_grids = len(grid_ids)
print('n_weeks:', n_weeks, 'n_grids:', n_grids)

# Event matrix uses target_col, not outbreak_count, to match 06 target.
ev = df.loc[df[target_col] == 1, ['week_start', 'grid_id']].copy()
ev['w_idx'] = ev['week_start'].map(week_index)
ev['g_idx'] = ev['grid_id'].map(grid_index)

event_matrix = np.zeros((n_weeks, n_grids), dtype=np.float32)
for r in ev.itertuples(index=False):
    if pd.notna(r.w_idx) and pd.notna(r.g_idx):
        event_matrix[int(r.w_idx), int(r.g_idx)] = 1.0

print('positive cells in event_matrix:', int((event_matrix > 0).sum()))

# Same-grid past features: previous k weeks including all weeks from t-k to t-1
created_features = []
same_grid_features = {}
for w in WINDOWS_WEEKS:
    arr = np.zeros_like(event_matrix, dtype=np.float32)
    # cumsum for efficient past window
    cs = np.vstack([np.zeros((1, n_grids), dtype=np.float32), np.cumsum(event_matrix, axis=0)])
    # arr[t] = sum event_matrix[t-w:t]
    for t in range(n_weeks):
        start = max(0, t - w)
        arr[t, :] = cs[t, :] - cs[start, :]
    col = f'past_same_grid_{w}w_count_v4'
    same_grid_features[col] = arr
    created_features.append(col)

# Neighbor adjacency matrices
coords_rad = np.deg2rad(grid[[lat_col, lon_col]].astype(float).values)
tree = BallTree(coords_rad, metric='haversine')
earth_radius_km = 6371.0088

neighbor_mats = {}
for radius in RADII_KM:
    ind = tree.query_radius(coords_rad, r=radius / earth_radius_km)
    rows, cols = [], []
    for i, neigh in enumerate(ind):
        # 自分自身を除外
        neigh = [j for j in neigh if j != i]
        rows.extend([i] * len(neigh))
        cols.extend(neigh)
    data = np.ones(len(rows), dtype=np.float32)
    mat = sparse.csr_matrix((data, (rows, cols)), shape=(n_grids, n_grids), dtype=np.float32)
    neighbor_mats[radius] = mat
    print(f'neighbor radius {radius} km: nonzero links = {mat.nnz:,}')

neighbor_features = {}
for radius, mat in neighbor_mats.items():
    for w in WINDOWS_WEEKS:
        # Same-grid past window matrix: week x grid
        base_col = f'past_same_grid_{w}w_count_v4'
        same_arr = same_grid_features[base_col]
        # neighbor count for each grid = sum over neighbor grids
        # same_arr is week x grid, mat is grid x grid, using same_arr @ mat.T
        neigh_arr = same_arr @ mat.T
        col = f'past_neighbor_{radius}km_{w}w_count_v4'
        neighbor_features[col] = np.asarray(neigh_arr, dtype=np.float32)
        created_features.append(col)

# Attach features by mapping week/grid indices
df['_w_idx'] = df['week_start'].map(week_index).astype(int)
df['_g_idx'] = df['grid_id'].map(grid_index).astype(int)

for col, arr in {**same_grid_features, **neighbor_features}.items():
    df[col] = arr[df['_w_idx'].values, df['_g_idx'].values].astype('float32')

df = df.drop(columns=['_w_idx', '_g_idx'])

feature_list_path = f'{RESULT_DIR}/09_created_feature_list_v4_strict_06_compatible.csv'
pd.DataFrame({'feature': created_features}).to_csv(feature_list_path, index=False, encoding='utf-8-sig')
print('created features:', len(created_features))
print('saved:', feature_list_path)
print(created_features[:20])

# 保存
panel_out = f'{PROC_DIR}/hpai_weekly_grid_panel_with_spatiotemporal_features_v4_strict_06_compatible.parquet'
df.to_parquet(panel_out, index=False)
print('saved panel:', panel_out, df.shape)

gc.collect()

n_weeks: 299 n_grids: 5491
positive cells in event_matrix: 191
neighbor radius 30 km: nonzero links = 205,410
neighbor radius 50 km: nonzero links = 523,462
created features: 15
saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/09_created_feature_list_v4_strict_06_compatible.csv
['past_same_grid_1w_count_v4', 'past_same_grid_2w_count_v4', 'past_same_grid_4w_count_v4', 'past_same_grid_8w_count_v4', 'past_same_grid_12w_count_v4', 'past_neighbor_30km_1w_count_v4', 'past_neighbor_30km_2w_count_v4', 'past_neighbor_30km_4w_count_v4', 'past_neighbor_30km_8w_count_v4', 'past_neighbor_30km_12w_count_v4', 'past_neighbor_50km_1w_count_v4', 'past_neighbor_50km_2w_count_v4', 'past_neighbor_50km_4w_count_v4', 'past_neighbor_50km_8w_count_v4', 'past_neighbor_50km_12w_count_v4']
saved panel: /content/drive/MyDrive/avian_influenza_project/processed/hpai_weekly_grid_panel_with_spatiotemporal_features_v4_strict_06_compatible.parquet (1641809, 49)


0

In [6]:
# ============================================================
# 5. Leakage-safe feature definitions, strictly aligned with 06
# ============================================================
weather_cols = [c for c in ['temp_mean_c', 'temp_min_c', 'temp_max_c'] if c in df.columns]
geo_cols = [c for c in [lon_col, lat_col] if c is not None]
season_cols = [c for c in ['weekofyear', 'month', 'sin_week', 'cos_week'] if c in df.columns]

# 06由来の既存ラグ特徴量があれば使う
lag_cols_06 = [
    c for c in ['lag_outbreak_1w', 'lag_outbreak_2w', 'lag_outbreak_4w', 'lag_outbreak_8w']
    if c in df.columns
]

neighbor_past_cols_06 = [
    c for c in [
        'neighbor_outbreak_count_past_1w',
        'neighbor_outbreak_count_past_2w',
        'neighbor_outbreak_count_past_4w',
        'neighbor_outbreak_count_past_8w',
    ] if c in df.columns
]

same_grid_v4_cols = [c for c in created_features if c.startswith('past_same_grid_')]
neighbor_v4_cols = [c for c in created_features if c.startswith('past_neighbor_')]

# 明確なリーク列・同週発生代理列を除外
force_exclude_cols = set([
    'outbreak_binary',
    'outbreak_count',
    'target',
    'num_birds_sum',
    'y_lead_1w',
    'y_lead_2w',
    'y_lead_4w',
    'neighbor_outbreak_binary',
    'neighbor_outbreak_count',
    'rolling_outbreak_4w',
    'rolling_outbreak_8w',
    'rolling_outbreak_12w',
])

def is_forbidden(c):
    if c in force_exclude_cols:
        return True
    if c.startswith('y_lead'):
        return True
    return False

def safe(cols):
    out = sorted(set([c for c in cols if c in df.columns and not is_forbidden(c)]))
    bad = [c for c in out if is_forbidden(c)]
    if bad:
        raise ValueError(f'forbidden features found: {bad}')
    return out

feature_cols_minimal = safe(weather_cols + geo_cols + season_cols)
feature_cols_lag06 = safe(feature_cols_minimal + lag_cols_06)
feature_cols_neighbor06 = safe(feature_cols_minimal + neighbor_past_cols_06)
feature_cols_lag_neighbor06 = safe(feature_cols_minimal + lag_cols_06 + neighbor_past_cols_06)
feature_cols_v4_same = safe(feature_cols_minimal + same_grid_v4_cols)
feature_cols_v4_neighbor = safe(feature_cols_minimal + neighbor_v4_cols)
feature_cols_v4_all = safe(feature_cols_minimal + same_grid_v4_cols + neighbor_v4_cols)

feature_sets = {
    'minimal_weather_geo_season_v4': feature_cols_minimal,
    'minimal_plus_lag06_v4': feature_cols_lag06,
    'minimal_plus_neighbor06_v4': feature_cols_neighbor06,
    'minimal_plus_lag_neighbor06_v4': feature_cols_lag_neighbor06,
    'minimal_plus_same_grid_past_v4': feature_cols_v4_same,
    'minimal_plus_neighbor_past_v4': feature_cols_v4_neighbor,
    'minimal_plus_spatiotemporal_past_v4': feature_cols_v4_all,
}

for name, cols in feature_sets.items():
    print(name, 'n_features:', len(cols))
    print(cols[:40])

pd.DataFrame([
    {'model_name': name, 'n_features': len(cols), 'features': ','.join(cols)}
    for name, cols in feature_sets.items()
]).to_csv(f'{RESULT_DIR}/09_feature_sets_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')

minimal_weather_geo_season_v4 n_features: 9
['cos_week', 'grid_lat', 'grid_lon', 'month', 'sin_week', 'temp_max_c', 'temp_mean_c', 'temp_min_c', 'weekofyear']
minimal_plus_lag06_v4 n_features: 13
['cos_week', 'grid_lat', 'grid_lon', 'lag_outbreak_1w', 'lag_outbreak_2w', 'lag_outbreak_4w', 'lag_outbreak_8w', 'month', 'sin_week', 'temp_max_c', 'temp_mean_c', 'temp_min_c', 'weekofyear']
minimal_plus_neighbor06_v4 n_features: 13
['cos_week', 'grid_lat', 'grid_lon', 'month', 'neighbor_outbreak_count_past_1w', 'neighbor_outbreak_count_past_2w', 'neighbor_outbreak_count_past_4w', 'neighbor_outbreak_count_past_8w', 'sin_week', 'temp_max_c', 'temp_mean_c', 'temp_min_c', 'weekofyear']
minimal_plus_lag_neighbor06_v4 n_features: 17
['cos_week', 'grid_lat', 'grid_lon', 'lag_outbreak_1w', 'lag_outbreak_2w', 'lag_outbreak_4w', 'lag_outbreak_8w', 'month', 'neighbor_outbreak_count_past_1w', 'neighbor_outbreak_count_past_2w', 'neighbor_outbreak_count_past_4w', 'neighbor_outbreak_count_past_8w', 'sin_wee

In [7]:
# ============================================================
# 6. Rolling split definitions: exactly same as 06
# ============================================================
rolling_splits = [
    {
        'split_name': 'test_fy2023',
        'train_start': pd.Timestamp('2020-08-01'),
        'train_end': pd.Timestamp('2023-04-01'),
        'test_start': pd.Timestamp('2023-04-01'),
        'test_end': pd.Timestamp('2024-04-01'),
    },
    {
        'split_name': 'test_fy2024',
        'train_start': pd.Timestamp('2020-08-01'),
        'train_end': pd.Timestamp('2024-04-01'),
        'test_start': pd.Timestamp('2024-04-01'),
        'test_end': pd.Timestamp('2025-04-01'),
    },
    {
        'split_name': 'test_fy2025',
        'train_start': pd.Timestamp('2020-08-01'),
        'train_end': pd.Timestamp('2025-04-01'),
        'test_start': pd.Timestamp('2025-04-01'),
        'test_end': pd.Timestamp('2026-04-01'),
    },
]

split_check_rows = []
for sp in rolling_splits:
    train_mask = (df['week_start'] >= sp['train_start']) & (df['week_start'] < sp['train_end'])
    test_mask = (df['week_start'] >= sp['test_start']) & (df['week_start'] < sp['test_end'])
    tr = df.loc[train_mask]
    te = df.loc[test_mask]
    split_check_rows.append({
        'split_name': sp['split_name'],
        'train_start': sp['train_start'].date(),
        'train_end_exclusive': sp['train_end'].date(),
        'test_start': sp['test_start'].date(),
        'test_end_exclusive': sp['test_end'].date(),
        'train_rows': len(tr),
        'train_events': int(tr[target_col].sum()) if len(tr) else 0,
        'test_rows': len(te),
        'test_events': int(te[target_col].sum()) if len(te) else 0,
        'test_event_rate': float(te[target_col].mean()) if len(te) else np.nan,
    })

split_check = pd.DataFrame(split_check_rows)
display(split_check)
split_check.to_csv(f'{RESULT_DIR}/09_rolling_split_check_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')

,split_name,train_start,train_end_exclusive,test_start,test_end_exclusive,train_rows,train_events,test_rows,test_events,test_event_rate
0,test_fy2023,2020-08-01,2023-04-01,2023-04-01,2024-04-01,741285,129,285532,11,0.000039
1,test_fy2024,2020-08-01,2024-04-01,2024-04-01,2025-04-01,1026817,140,291023,30,0.000103
2,test_fy2025,2020-08-01,2025-04-01,2025-04-01,2026-04-01,1317840,170,285532,20,0.000070


In [8]:
# ============================================================
# 7. Training and rolling risk-map evaluation functions
# ============================================================
def add_weekly_risk_ranks(result):
    out = result.copy()
    out['n_grids_in_week'] = out.groupby('week_start')['grid_id'].transform('count')
    out['risk_rank'] = out.groupby('week_start')['pred_proba'].rank(
        method='first', ascending=False
    )
    out['risk_percentile'] = 1.0 - ((out['risk_rank'] - 1.0) / out['n_grids_in_week'])

    for p in [1, 5, 10, 20]:
        cutoff = np.ceil(out['n_grids_in_week'] * p / 100.0)
        out[f'top{p}'] = out['risk_rank'] <= cutoff

    return out

def summarize_topk(result, split_info, model_name):
    event_rows = result[result[target_col] == 1].copy()
    n_events = len(event_rows)

    row = {
        'split_name': split_info['split_name'],
        'model_name': model_name,
        'train_start': str(split_info['train_start'].date()),
        'train_end_exclusive': str(split_info['train_end'].date()),
        'test_start': str(split_info['test_start'].date()),
        'test_end_exclusive': str(split_info['test_end'].date()),
        'test_rows': len(result),
        'n_events': n_events,
        'baseline_event_rate': float(result[target_col].mean()) if len(result) else np.nan,
    }

    if n_events == 0:
        for p in [1, 5, 10, 20]:
            row[f'top{p}_events'] = 0
            row[f'top{p}_capture_rate'] = np.nan
        row['mean_event_percentile'] = np.nan
        row['median_event_percentile'] = np.nan
        row['mean_event_rank'] = np.nan
        row['median_event_rank'] = np.nan
        return row

    for p in [1, 5, 10, 20]:
        row[f'top{p}_events'] = int(event_rows[f'top{p}'].sum())
        row[f'top{p}_capture_rate'] = float(event_rows[f'top{p}'].mean())

    row['mean_event_percentile'] = float(event_rows['risk_percentile'].mean())
    row['median_event_percentile'] = float(event_rows['risk_percentile'].median())
    row['mean_event_rank'] = float(event_rows['risk_rank'].mean())
    row['median_event_rank'] = float(event_rows['risk_rank'].median())
    return row

def train_eval_one_split_model(train_df, test_df, feature_cols, split_info, model_name, random_state=42):
    if len(feature_cols) == 0:
        raise ValueError(f'{model_name}: feature_cols が空です．')

    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].astype(int).copy()
    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].astype(int).copy()

    if y_train.nunique() < 2:
        raise ValueError(f'{split_info["split_name"]} / {model_name}: 学習データに片方のクラスしかありません．')

    med = X_train.median(numeric_only=True)
    X_train = X_train.fillna(med)
    X_test = X_test.fillna(med)

    rf = RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        class_weight='balanced',
        random_state=random_state,
        n_jobs=N_JOBS,
    )
    rf.fit(X_train, y_train)
    pred_proba = rf.predict_proba(X_test)[:, 1]

    keep_cols = ['grid_id', 'week_start', target_col]
    if 'outbreak_count' in test_df.columns:
        keep_cols.append('outbreak_count')

    result = test_df[keep_cols].copy()
    result['pred_proba'] = pred_proba
    result['model_name'] = model_name
    result['split_name'] = split_info['split_name']
    result = add_weekly_risk_ranks(result)

    if y_test.nunique() == 2:
        roc_auc = roc_auc_score(y_test, pred_proba)
        pr_auc = average_precision_score(y_test, pred_proba)
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    topk_row = summarize_topk(result, split_info, model_name)
    topk_row['roc_auc'] = float(roc_auc) if not pd.isna(roc_auc) else np.nan
    topk_row['pr_auc'] = float(pr_auc) if not pd.isna(pr_auc) else np.nan
    if topk_row['baseline_event_rate'] and topk_row['baseline_event_rate'] > 0 and not pd.isna(topk_row['pr_auc']):
        topk_row['pr_auc_over_baseline'] = topk_row['pr_auc'] / topk_row['baseline_event_rate']
    else:
        topk_row['pr_auc_over_baseline'] = np.nan
    topk_row['n_features'] = len(feature_cols)

    importance = pd.DataFrame({
        'split_name': split_info['split_name'],
        'model_name': model_name,
        'feature': feature_cols,
        'importance': rf.feature_importances_,
    }).sort_values('importance', ascending=False)

    pred_file = f'{RESULT_DIR}/09_rolling_{split_info["split_name"]}_{model_name}_predictions_v4_strict_06_compatible.csv'
    result.to_csv(pred_file, index=False, encoding='utf-8-sig')

    model_file = f'{MODEL_DIR}/09_rolling_{split_info["split_name"]}_{model_name}_v4_strict_06_compatible.joblib'
    joblib.dump(rf, model_file)

    return rf, result, topk_row, importance

In [9]:
# ============================================================
# 8. Run rolling evaluation
# ============================================================
all_summary_rows = []
all_case_results = []
all_importances = []

for sp in rolling_splits:
    print('\n' + '=' * 100)
    print('SPLIT:', sp['split_name'])
    print('train:', sp['train_start'].date(), 'to', sp['train_end'].date(), '(exclusive)')
    print('test :', sp['test_start'].date(), 'to', sp['test_end'].date(), '(exclusive)')

    train_mask = (df['week_start'] >= sp['train_start']) & (df['week_start'] < sp['train_end'])
    test_mask = (df['week_start'] >= sp['test_start']) & (df['week_start'] < sp['test_end'])
    train_df = df.loc[train_mask].copy()
    test_df = df.loc[test_mask].copy()

    print('train rows/events:', len(train_df), int(train_df[target_col].sum()))
    print('test rows/events :', len(test_df), int(test_df[target_col].sum()))

    if len(train_df) == 0 or len(test_df) == 0:
        print('データが不足しているため，このsplitをスキップします．')
        continue
    if train_df[target_col].nunique() < 2:
        print('学習データに片方のクラスしかないため，このsplitをスキップします．')
        continue

    for model_name, feature_cols in feature_sets.items():
        print('\n' + '-' * 80)
        print('MODEL:', model_name)
        print('n_features:', len(feature_cols))
        try:
            rf, result, summary_row, importance = train_eval_one_split_model(
                train_df=train_df,
                test_df=test_df,
                feature_cols=feature_cols,
                split_info=sp,
                model_name=model_name,
                random_state=RANDOM_STATE,
            )
            all_summary_rows.append(summary_row)
            all_case_results.append(result[result[target_col] == 1].copy())
            all_importances.append(importance)

            print('ROC-AUC:', summary_row['roc_auc'])
            print('PR-AUC:', summary_row['pr_auc'])
            print('top10_capture_rate:', summary_row['top10_capture_rate'])
            print('top20_capture_rate:', summary_row['top20_capture_rate'])
            print('median_event_percentile:', summary_row['median_event_percentile'])
        except Exception as e:
            print('ERROR:', repr(e))
            continue

    del train_df, test_df
    gc.collect()

summary_results = pd.DataFrame(all_summary_rows)
case_results = pd.concat(all_case_results, ignore_index=True) if all_case_results else pd.DataFrame()
importance_all = pd.concat(all_importances, ignore_index=True) if all_importances else pd.DataFrame()

print('\nsummary_results')
display(summary_results)

summary_results.to_csv(f'{RESULT_DIR}/09_rolling_model_summary_by_split_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')
case_results.to_csv(f'{RESULT_DIR}/09_rolling_outbreak_case_results_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')
importance_all.to_csv(f'{RESULT_DIR}/09_rolling_feature_importance_all_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')


SPLIT: test_fy2023
train: 2020-08-01 to 2023-04-01 (exclusive)
test : 2023-04-01 to 2024-04-01 (exclusive)
train rows/events: 741285 129
test rows/events : 285532 11

--------------------------------------------------------------------------------
MODEL: minimal_weather_geo_season_v4
n_features: 9
ROC-AUC: 0.8723579001194307
PR-AUC: 0.00048529228076983383
top10_capture_rate: 0.2727272727272727
top20_capture_rate: 0.7272727272727273
median_event_percentile: 0.8175195774904389

--------------------------------------------------------------------------------
MODEL: minimal_plus_lag06_v4
n_features: 13
ROC-AUC: 0.8953205479870767
PR-AUC: 0.00040030488778186656
top10_capture_rate: 0.2727272727272727
top20_capture_rate: 0.45454545454545453
median_event_percentile: 0.7836459661263886

--------------------------------------------------------------------------------
MODEL: minimal_plus_neighbor06_v4
n_features: 13
ROC-AUC: 0.8831620727785984
PR-AUC: 0.003344851581761035
top10_capture_rate: 0.2

,split_name,model_name,train_start,train_end_exclusive,test_start,test_end_exclusive,test_rows,n_events,baseline_event_rate,top1_events,...,top20_events,top20_capture_rate,mean_event_percentile,median_event_percentile,mean_event_rank,median_event_rank,roc_auc,pr_auc,pr_auc_over_baseline,n_features
0,test_fy2023,minimal_weather_geo_season_v4,2020-08-01,2023-04-01,2023-04-01,2024-04-01,285532,11,0.000039,2,...,8,0.727273,0.800566,0.817520,1096.090909,1003.0,0.872358,0.000485,12.596952,9
1,test_fy2023,minimal_plus_lag06_v4,2020-08-01,2023-04-01,2023-04-01,2024-04-01,285532,11,0.000039,2,...,5,0.454545,0.782702,0.783646,1194.181818,1189.0,0.895321,0.000400,10.390896,13
2,test_fy2023,minimal_plus_neighbor06_v4,2020-08-01,2023-04-01,2023-04-01,2024-04-01,285532,11,0.000039,2,...,4,0.363636,0.778679,0.757057,1216.272727,1335.0,0.883162,0.003345,86.823833,13
3,test_fy2023,minimal_plus_lag_neighbor06_v4,2020-08-01,2023-04-01,2023-04-01,2024-04-01,285532,11,0.000039,2,...,5,0.454545,0.792752,0.758696,1139.000000,1326.0,0.888459,0.002202,57.151061,17
4,test_fy2023,minimal_plus_same_grid_past_v4,2020-08-01,2023-04-01,2023-04-01,2024-04-01,285532,11,0.000039,1,...,4,0.363636,0.780633,0.773083,1205.545455,1247.0,0.893350,0.000236,6.114780,14
5,test_fy2023,minimal_plus_neighbor_past_v4,2020-08-01,2023-04-01,2023-04-01,2024-04-01,285532,11,0.000039,2,...,3,0.272727,0.793248,0.792388,1136.272727,1141.0,0.883684,0.011591,300.870824,19
6,test_fy2023,minimal_plus_spatiotemporal_past_v4,2020-08-01,2023-04-01,2023-04-01,2024-04-01,285532,11,0.000039,1,...,6,0.545455,0.781709,0.808960,1199.636364,1050.0,0.880210,0.000296,7.682880,24
7,test_fy2024,minimal_weather_geo_season_v4,2020-08-01,2024-04-01,2024-04-01,2025-04-01,291023,30,0.000103,3,...,14,0.466667,0.661586,0.783646,1859.233333,1189.0,0.803073,0.001649,15.993122,9
8,test_fy2024,minimal_plus_lag06_v4,2020-08-01,2024-04-01,2024-04-01,2025-04-01,291023,30,0.000103,3,...,17,0.566667,0.701505,0.808778,1640.033333,1051.0,0.845618,0.001099,10.658423,13
9,test_fy2024,minimal_plus_neighbor06_v4,2020-08-01,2024-04-01,2024-04-01,2025-04-01,291023,30,0.000103,2,...,17,0.566667,0.690931,0.854762,1698.100000,798.5,0.805004,0.002447,23.733589,13


In [10]:
# ============================================================
# 9. Summary by model and comparison with 07
# ============================================================
if len(summary_results) == 0:
    raise ValueError('summary_results が空です．前セルのエラーを確認してください．')

summary_by_model = (summary_results
    .groupby('model_name', as_index=False)
    .agg(
        n_splits=('split_name', 'nunique'),
        n_events=('n_events', 'sum'),
        mean_top1_capture_rate=('top1_capture_rate', 'mean'),
        mean_top5_capture_rate=('top5_capture_rate', 'mean'),
        mean_top10_capture_rate=('top10_capture_rate', 'mean'),
        mean_top20_capture_rate=('top20_capture_rate', 'mean'),
        mean_event_percentile=('mean_event_percentile', 'mean'),
        median_event_percentile=('median_event_percentile', 'mean'),
        mean_event_rank=('mean_event_rank', 'mean'),
        roc_auc=('roc_auc', 'mean'),
        pr_auc=('pr_auc', 'mean'),
        pr_auc_over_baseline=('pr_auc_over_baseline', 'mean'),
        n_features=('n_features', 'mean')
    )
    .sort_values(['mean_top10_capture_rate', 'mean_event_percentile'], ascending=False)
)

display(summary_by_model)
summary_by_model.to_csv(f'{RESULT_DIR}/09_rolling_model_summary_by_model_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')

# 07との比較
summary_07_path = f'{RESULT_DIR}/07_topk_hit_summary_by_model.csv'
if os.path.exists(summary_07_path):
    s07 = pd.read_csv(summary_07_path)
    s07['source'] = '07'
    s09 = summary_by_model.copy()
    s09['source'] = '09_v4'
    compare = pd.concat([s07, s09], ignore_index=True, sort=False)
    display(compare)
    compare.to_csv(f'{RESULT_DIR}/09_compare_with_07_model_summary_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')
else:
    print('07 summary not found:', summary_07_path)

# 特徴量重要度
if len(importance_all) > 0:
    fi_mean = (importance_all
        .groupby(['model_name', 'feature'], as_index=False)['importance'].mean()
        .sort_values(['model_name', 'importance'], ascending=[True, False]))
    display(fi_mean.groupby('model_name').head(20))
    fi_mean.to_csv(f'{RESULT_DIR}/09_feature_importance_mean_by_model_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')

,model_name,n_splits,n_events,mean_top1_capture_rate,mean_top5_capture_rate,mean_top10_capture_rate,mean_top20_capture_rate,mean_event_percentile,median_event_percentile,mean_event_rank,roc_auc,pr_auc,pr_auc_over_baseline,n_features
4,minimal_plus_same_grid_past_v4,3,61,0.063636,0.146970,0.340909,0.493434,0.750376,0.800188,1371.687374,0.877515,0.000510,6.885069,14.0
1,minimal_plus_lag_neighbor06_v4,3,61,0.105051,0.177273,0.329798,0.484848,0.753094,0.794603,1356.761111,0.866309,0.001166,23.870208,17.0
2,minimal_plus_neighbor06_v4,3,61,0.082828,0.157576,0.329798,0.510101,0.745251,0.813209,1399.824242,0.854223,0.002068,38.818467,13.0
5,minimal_plus_spatiotemporal_past_v4,3,61,0.102525,0.169192,0.302020,0.531818,0.746508,0.811995,1392.923232,0.862727,0.000576,7.824005,24.0
0,minimal_plus_lag06_v4,3,61,0.093939,0.168687,0.268687,0.557071,0.745756,0.805955,1397.055051,0.871886,0.000636,8.968178,13.0
3,minimal_plus_neighbor_past_v4,3,61,0.082828,0.218687,0.263131,0.474242,0.744287,0.807564,1405.118687,0.849692,0.004626,108.299946,19.0
6,minimal_weather_geo_season_v4,3,61,0.093939,0.132828,0.240909,0.614646,0.739600,0.818946,1430.858081,0.849522,0.000866,11.736913,9.0


,model_name,n_splits,total_events,top1_capture_rate_mean,top5_capture_rate_mean,top10_capture_rate_mean,top20_capture_rate_mean,mean_event_percentile_mean,median_event_percentile_mean,mean_event_rank_mean,...,mean_top5_capture_rate,mean_top10_capture_rate,mean_top20_capture_rate,mean_event_percentile,median_event_percentile,mean_event_rank,roc_auc,pr_auc,pr_auc_over_baseline,n_features
0,minimal_plus_neighbor_past,3,61.0,0.082828,0.157576,0.329798,0.510101,0.745251,0.813209,1399.824242,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,minimal_plus_lag_neighbor_past,3,61.0,0.105051,0.177273,0.329798,0.484848,0.753094,0.794603,1356.761111,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,minimal_plus_lag,3,61.0,0.093939,0.168687,0.268687,0.557071,0.745756,0.805955,1397.055051,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,minimal_weather_geo_season,3,61.0,0.093939,0.132828,0.240909,0.614646,0.739600,0.818946,1430.858081,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,minimal_plus_same_grid_past_v4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.146970,0.340909,0.493434,0.750376,0.800188,1371.687374,0.877515,0.000510,6.885069,14.0
5,minimal_plus_lag_neighbor06_v4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.177273,0.329798,0.484848,0.753094,0.794603,1356.761111,0.866309,0.001166,23.870208,17.0
6,minimal_plus_neighbor06_v4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.157576,0.329798,0.510101,0.745251,0.813209,1399.824242,0.854223,0.002068,38.818467,13.0
7,minimal_plus_spatiotemporal_past_v4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.169192,0.302020,0.531818,0.746508,0.811995,1392.923232,0.862727,0.000576,7.824005,24.0
8,minimal_plus_lag06_v4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.168687,0.268687,0.557071,0.745756,0.805955,1397.055051,0.871886,0.000636,8.968178,13.0
9,minimal_plus_neighbor_past_v4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.218687,0.263131,0.474242,0.744287,0.807564,1405.118687,0.849692,0.004626,108.299946,19.0


,model_name,feature,importance
0,minimal_plus_lag06_v4,cos_week,0.172357
10,minimal_plus_lag06_v4,temp_mean_c,0.146456
9,minimal_plus_lag06_v4,temp_max_c,0.134716
11,minimal_plus_lag06_v4,temp_min_c,0.129429
1,minimal_plus_lag06_v4,grid_lat,0.119746
...,...,...,...
107,minimal_weather_geo_season_v4,temp_min_c,0.120268
102,minimal_weather_geo_season_v4,grid_lon,0.111949
108,minimal_weather_geo_season_v4,weekofyear,0.072043
104,minimal_weather_geo_season_v4,sin_week,0.049888


In [11]:
# ============================================================
# 10. Extract hit/miss outbreak cases for the best model
# ============================================================
if len(case_results) == 0:
    print('発生地点ケースがありません．')
else:
    best_model = summary_by_model.iloc[0]['model_name']
    print('best_model:', best_model)

    best_cases = case_results[case_results['model_name'] == best_model].copy()
    hit_top10 = best_cases[best_cases['top10'] == True].copy()
    miss_top10 = best_cases[best_cases['top10'] == False].copy()

    print('all event cases:', best_cases.shape)
    print('hit top10:', hit_top10.shape)
    print('miss top10:', miss_top10.shape)

    best_cases.to_csv(f'{RESULT_DIR}/09_best_model_event_cases_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')
    hit_top10.to_csv(f'{RESULT_DIR}/09_best_model_top10_hit_events_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')
    miss_top10.to_csv(f'{RESULT_DIR}/09_best_model_missed_top10_events_v4_strict_06_compatible.csv', index=False, encoding='utf-8-sig')

    display(hit_top10.sort_values('risk_percentile', ascending=False).head(30))
    display(miss_top10.sort_values('risk_percentile', ascending=True).head(30))

best_model: minimal_plus_same_grid_past_v4
all event cases: (61, 14)
hit top10: (22, 14)
miss top10: (39, 14)


,grid_id,week_start,outbreak_binary,outbreak_count,pred_proba,model_name,split_name,n_grids_in_week,risk_rank,risk_percentile,top1,top5,top10,top20
197,G000508,2024-11-18,1,1,0.506334,minimal_plus_same_grid_past_v4,test_fy2024,5491,4.0,0.999454,True,True,True,True
205,G001842,2024-11-04,1,1,0.667622,minimal_plus_same_grid_past_v4,test_fy2024,5491,5.0,0.999272,True,True,True,True
54,G005326,2023-04-03,1,2,0.321669,minimal_plus_same_grid_past_v4,test_fy2023,5491,8.0,0.998725,True,True,True,True
218,G004430,2025-01-27,1,2,0.741691,minimal_plus_same_grid_past_v4,test_fy2024,5491,32.0,0.994354,True,True,True,True
204,G001621,2024-12-16,1,1,0.750827,minimal_plus_same_grid_past_v4,test_fy2024,5491,91.0,0.983610,False,True,True,True
379,G004117,2025-12-22,1,1,0.561621,minimal_plus_same_grid_past_v4,test_fy2025,5491,136.0,0.975414,False,True,True,True
208,G002735,2025-01-06,1,5,0.555870,minimal_plus_same_grid_past_v4,test_fy2024,5491,149.0,0.973047,False,True,True,True
200,G000933,2025-01-06,1,1,0.523617,minimal_plus_same_grid_past_v4,test_fy2024,5491,170.0,0.969222,False,True,True,True
210,G002735,2025-01-20,1,1,0.611118,minimal_plus_same_grid_past_v4,test_fy2024,5491,178.0,0.967765,False,True,True,True
209,G002735,2025-01-13,1,2,0.557439,minimal_plus_same_grid_past_v4,test_fy2024,5491,219.0,0.960299,False,True,True,True


,grid_id,week_start,outbreak_binary,outbreak_count,pred_proba,model_name,split_name,n_grids_in_week,risk_rank,risk_percentile,top1,top5,top10,top20
217,G004107,2024-04-29,1,1,0.003024,minimal_plus_same_grid_past_v4,test_fy2024,5491,5421.0,0.012930,False,False,False,False
219,G004433,2024-10-21,1,1,0.001622,minimal_plus_same_grid_past_v4,test_fy2024,5491,5404.0,0.016026,False,False,False,False
226,G005825,2024-11-11,1,1,0.002307,minimal_plus_same_grid_past_v4,test_fy2024,5491,5256.0,0.042979,False,False,False,False
386,G005504,2025-12-29,1,1,0.029461,minimal_plus_same_grid_past_v4,test_fy2025,5491,4246.0,0.226917,False,False,False,False
221,G005119,2024-12-30,1,1,0.004353,minimal_plus_same_grid_past_v4,test_fy2024,5491,4117.0,0.250410,False,False,False,False
383,G005272,2025-10-27,1,1,0.063405,minimal_plus_same_grid_past_v4,test_fy2025,5491,3965.0,0.278091,False,False,False,False
224,G005259,2024-12-30,1,1,0.006092,minimal_plus_same_grid_past_v4,test_fy2024,5491,3801.0,0.307958,False,False,False,False
385,G005502,2026-03-02,1,1,0.006896,minimal_plus_same_grid_past_v4,test_fy2025,5491,3707.0,0.325077,False,False,False,False
213,G003074,2024-10-21,1,1,0.004300,minimal_plus_same_grid_past_v4,test_fy2024,5491,3226.0,0.412675,False,False,False,False
51,G003237,2024-01-01,1,1,0.095370,minimal_plus_same_grid_past_v4,test_fy2023,5491,2601.0,0.526498,False,False,False,False


In [12]:
# ============================================================
# 11. Write summary report
# ============================================================
best = summary_by_model.iloc[0].to_dict()

report = f'''# 09 Summary: strict 06-compatible spatiotemporal features v4

This report is generated by `{NOTEBOOK_VERSION}`.

## Key leakage controls
- The target variable is the same as 06: `{target_col}`.
- `num_birds_sum` is excluded because it behaved as a same-week proxy and caused unrealistic perfect prediction in v3.
- `outbreak_count`, `outbreak_binary`, `target`, and `y_lead_*` are excluded from explanatory features.
- Rolling splits are the same fiscal-year splits used in 06.
- Added spatiotemporal features use only weeks before the target week.

## Best model
- Best model: `{best.get('model_name')}`
- Mean Top 1% capture rate: {best.get('mean_top1_capture_rate'):.3f}
- Mean Top 5% capture rate: {best.get('mean_top5_capture_rate'):.3f}
- Mean Top 10% capture rate: {best.get('mean_top10_capture_rate'):.3f}
- Mean Top 20% capture rate: {best.get('mean_top20_capture_rate'):.3f}
- Mean event percentile: {best.get('mean_event_percentile'):.3f}
- ROC-AUC: {best.get('roc_auc'):.3f}
- PR-AUC: {best.get('pr_auc'):.6f}

## Created v4 features
{chr(10).join('- ' + f for f in created_features)}

## Main output files
- `09_rolling_model_summary_by_split_v4_strict_06_compatible.csv`
- `09_rolling_model_summary_by_model_v4_strict_06_compatible.csv`
- `09_rolling_outbreak_case_results_v4_strict_06_compatible.csv`
- `09_feature_importance_mean_by_model_v4_strict_06_compatible.csv`
- `09_compare_with_07_model_summary_v4_strict_06_compatible.csv`
'''

report_path = f'{RESULT_DIR}/09_summary_report_v4_strict_06_compatible.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print('saved:', report_path)
print(report)

saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/09_summary_report_v4_strict_06_compatible.md
# 09 Summary: strict 06-compatible spatiotemporal features v4

This report is generated by `09_add_spatiotemporal_environment_features_same_paths_v5_auto_save_to_drive`.

## Key leakage controls
- The target variable is the same as 06: `outbreak_binary`.
- `num_birds_sum` is excluded because it behaved as a same-week proxy and caused unrealistic perfect prediction in v3.
- `outbreak_count`, `outbreak_binary`, `target`, and `y_lead_*` are excluded from explanatory features.
- Rolling splits are the same fiscal-year splits used in 06.
- Added spatiotemporal features use only weeks before the target week.

## Best model
- Best model: `minimal_plus_same_grid_past_v4`
- Mean Top 1% capture rate: 0.064
- Mean Top 5% capture rate: 0.147
- Mean Top 10% capture rate: 0.341
- Mean Top 20% capture rate: 0.493
- Mean event percentile: 0.750
- ROC-AUC: 0.878
- PR-A

## 12. Final auto-save to Google Drive

実行後に生成された09系の出力ファイルをGoogle Drive上の最終出力フォルダへ集約し，zipファイルも作成する．


In [13]:
# ============================================================
# 12. Final auto-save to Google Drive
# ============================================================
# ここまでで各CSV/Parquet/モデル/レポートは RESULT_DIR に保存されている．
# このセルでは，最終的に確認・提出しやすいように，09系の出力ファイルを
# Google Drive上の FINAL_OUTPUT_DIR に自動集約し，zipにもまとめる．

import os
import glob
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

# Colab上でDriveが見えていない場合は再マウントを試みる．
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception as e:
    print('Drive remount skipped or failed:', repr(e))

# FINAL_OUTPUT_DIR が未定義の環境でも動くようにする．
if 'FINAL_OUTPUT_DIR' not in globals():
    FINAL_OUTPUT_DIR = f'{RESULT_DIR}/09_final_outputs_v5_auto_save_to_drive'

os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)

# 09系の主要出力を集約する．
# モデルファイルや予測ファイルを含めるため，拡張子は限定せず 09_ / 09- / NOTEBOOK_VERSION を拾う．
patterns = [
    f'{RESULT_DIR}/09_*',
    f'{RESULT_DIR}/{NOTEBOOK_VERSION}*',
]

src_files = []
for pat in patterns:
    src_files.extend(glob.glob(pat))

# final output folderや既存zip自体は二重コピーしない．
final_dir_resolved = Path(FINAL_OUTPUT_DIR).resolve()
unique_files = []
seen = set()
for f in src_files:
    p = Path(f)
    if not p.is_file():
        continue
    try:
        if final_dir_resolved in p.resolve().parents:
            continue
    except Exception:
        pass
    if p.name.endswith('.zip'):
        continue
    key = str(p.resolve())
    if key not in seen:
        seen.add(key)
        unique_files.append(p)

copied = []
for p in sorted(unique_files, key=lambda x: x.name):
    dst = Path(FINAL_OUTPUT_DIR) / p.name
    shutil.copy2(p, dst)
    copied.append(dst)

# 実行時刻と保存一覧をmanifestに残す．
manifest_path = Path(FINAL_OUTPUT_DIR) / '09_final_output_manifest_v5_auto_save_to_drive.txt'
with open(manifest_path, 'w', encoding='utf-8') as f:
    f.write(f'Notebook version: {NOTEBOOK_VERSION}\n')
    f.write(f'Generated at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
    f.write(f'Original RESULT_DIR: {RESULT_DIR}\n')
    f.write(f'Final Google Drive folder: {FINAL_OUTPUT_DIR}\n')
    f.write(f'Number of copied files: {len(copied)}\n\n')
    for p in copied:
        f.write(str(p) + '\n')

# 最終出力フォルダをzip化する．
zip_path = Path(RESULT_DIR) / '09_final_outputs_v5_auto_save_to_drive.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(FINAL_OUTPUT_DIR).glob('*')):
        if p.is_file():
            zf.write(p, arcname=p.name)

# zipも最終フォルダ内にコピーしておく．
zip_in_final = Path(FINAL_OUTPUT_DIR) / zip_path.name
shutil.copy2(zip_path, zip_in_final)

print('Final outputs automatically saved to Google Drive:')
print(FINAL_OUTPUT_DIR)
print('\nCreated zip:')
print(zip_in_final)
print('\nCopied files:')
for p in copied:
    print('-', p.name)
print('-', manifest_path.name)
print('-', zip_in_final.name)


Final outputs automatically saved to Google Drive:
/content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/09_final_outputs_v5_auto_save_to_drive

Created zip:
/content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/09_final_outputs_v5_auto_save_to_drive/09_final_outputs_v5_auto_save_to_drive.zip

Copied files:
- 09_best_model_event_cases_v4_strict_06_compatible.csv
- 09_best_model_missed_top10_events_v4_strict_06_compatible.csv
- 09_best_model_top10_hit_events_v4_strict_06_compatible.csv
- 09_compare_with_07_model_summary_v4_strict_06_compatible.csv
- 09_created_feature_list_v4_strict_06_compatible.csv
- 09_feature_importance_mean_by_model_v4_strict_06_compatible.csv
- 09_feature_sets_v4_strict_06_compatible.csv
- 09_rolling_feature_importance_all_v4_strict_06_compatible.csv
- 09_rolling_model_summary_by_model_v4_strict_06_compatible.csv
- 09_rolling_model_summary_by_split_v4_strict_06_compatible.csv
- 09_rolling_outbreak_case_re